In [2]:
# ya ejecutado, NO VOLVER A EJECUTAR

# import sys
# !{sys.executable} -m pip install folium

In [6]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("F1_Preguntas")
    .enableHiveSupport()
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark.sql("USE f1_dw")

DataFrame[]

In [7]:
import folium
from branca.colormap import LinearColormap

# 1) Resultado para el mapa: todos los circuitos con >=5 carreras (sin LIMIT)
q2_mapa = spark.sql("""
SELECT c.circuitName AS circuito, 
       c.country     AS pais,
       c.latitude    AS lat, 
       c.longitude   AS lon,
       COUNT(DISTINCT f.raceId) AS carreras,
       SUM(CASE WHEN f.positionText = 'R' THEN 1 ELSE 0 END) AS dnf,
       ROUND(100.0 * SUM(CASE WHEN f.positionText = 'R' THEN 1 ELSE 0 END)
             / COUNT(*), 2) AS tasa_dnf_pct
FROM fact_results f
JOIN dim_race    r ON f.raceId    = r.raceId
JOIN dim_circuit c ON r.circuitId = c.circuitId
WHERE f.positionText <> 'W'
GROUP BY c.circuitId, c.circuitName, c.country, c.latitude, c.longitude
HAVING COUNT(DISTINCT f.raceId) >= 5
""")

In [8]:
# 2) A pandas (única parte con pandas, solo para visualizar)
df = q2_mapa.toPandas()
df["tasa_dnf_pct"] = df["tasa_dnf_pct"].astype(float)

In [15]:
# 3) Mapa
m = folium.Map(location=[20, 0], zoom_start=2, tiles="cartodbpositron")

cmap = LinearColormap(["green", "yellow", "red"],
                      vmin=df["tasa_dnf_pct"].min(),
                      vmax=df["tasa_dnf_pct"].max())
cmap.caption = "Tasa de DNF (%)"
m.add_child(cmap)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5 + row["carreras"] / 5,          # tamaño ~ cantidad de carreras
        color=cmap(row["tasa_dnf_pct"]),
        fill=True, fill_color=cmap(row["tasa_dnf_pct"]), fill_opacity=0.85,
        weight=1,
        tooltip=f'{row["circuito"]}: {row["tasa_dnf_pct"]}%',
        popup=folium.Popup(
            f'<b>{row["circuito"]}</b> ({row["pais"]})<br>'
            f'DNF: {row["tasa_dnf_pct"]}%<br>'
            f'{int(row["carreras"])} carreras', max_width=250),
    ).add_to(m)

m.save("mapa_dnf.html")

In [16]:
m   # renderizado